In [1]:
print("kernel alive")

kernel alive


In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import ElasticNetCV, ElasticNet
from sklearn.metrics import r2_score, mean_squared_error
from scipy.stats import pearsonr

In [12]:
data_file = Path("../data/processed/H13_promoters_651motifs_with_TPM.csv")

df = pd.read_csv(data_file)
df = df.dropna(subset=["log2_TPM_plus1"]).copy()

print(df.shape)
display(df.head())

(20007, 654)


,promoter_id,AHCTF1.H13CORE.0.B.B,AHR.H13CORE.0.P.B,ALX1.H13CORE.0.SM.B,ANDR.H13CORE.1.S.C,ANDR.H13CORE.2.P.B,AP2A.H13CORE.0.PSM.A,ARNT.H13CORE.0.P.B,ARNT2.H13CORE.0.P.B,ATF1.H13CORE.1.P.B,...,ZSC29.H13CORE.1.M.C,ZSC31.H13CORE.0.P.C,ZSCA1.H13CORE.0.SM.B,ZSCAN12.H13CORE.0.P.B,ZSCAN2.H13CORE.0.PG.A,ZSCAN25.H13CORE.0.PSG.A,ZXDA.H13CORE.0.PSI.A,ZXDC.H13CORE.0.PI.A,gene_symbol,log2_TPM_plus1
0,gene-OR4F16|OR4F16|chr1:107876-117877|-,302.332236,17.941461,34.013831,164.797344,335.398194,3.790701,48.721730,144.299281,18.516320,...,12.155462,35.184567,2.758547,138.829016,0.365287,7.726213,36.682951,15.609821,OR4F16,0.150560
1,gene-SAMD11|SAMD11|chr1:348565-358566|+,19.340017,106.962620,0.704667,67.601283,58.273912,126.511248,105.848932,225.198969,111.177175,...,2.670962,179.830138,9.959176,266.290223,0.000000,95.240338,400.288574,121.867755,SAMD11,5.361768
2,gene-NOC2L|NOC2L|chr1:383040-393041|-,43.043277,110.551691,0.630066,64.879750,106.249774,163.746683,107.512710,195.744209,120.124106,...,2.299665,180.647134,3.405136,276.570075,0.246302,67.752125,311.213273,105.019855,NOC2L,7.795195
3,gene-KLHL17|KLHL17|chr1:384373-394374|+,24.091429,99.126770,1.826247,59.478485,72.509248,194.859057,106.007893,190.749799,121.381597,...,2.205529,204.674574,3.148192,278.045765,0.380648,68.033367,350.141791,129.096476,KLHL17,4.693766
4,gene-PLEKHN1|PLEKHN1|chr1:390192-400193|+,28.317637,139.136414,5.733277,80.299276,137.485214,166.712558,175.525151,308.016942,80.768011,...,12.315560,193.470044,28.377810,255.382943,0.246302,89.286750,318.594021,135.428670,PLEKHN1,0.000000


In [13]:
target_col = "log2_TPM_plus1"

exclude_cols = [
    "promoter_id",
    "gene_symbol",
    target_col
]

feature_cols = [c for c in df.columns if c not in exclude_cols]

X = df[feature_cols]
y = df[target_col]

print("X:", X.shape)
print("y:", y.shape)
print("Number of features:", len(feature_cols))
print("First 10 features:", feature_cols[:10])

X: (20007, 651)
y: (20007,)
Number of features: 651
First 10 features: ['AHCTF1.H13CORE.0.B.B', 'AHR.H13CORE.0.P.B', 'ALX1.H13CORE.0.SM.B', 'ANDR.H13CORE.1.S.C', 'ANDR.H13CORE.2.P.B', 'AP2A.H13CORE.0.PSM.A', 'ARNT.H13CORE.0.P.B', 'ARNT2.H13CORE.0.P.B', 'ATF1.H13CORE.1.P.B', 'ATMIN.H13CORE.0.P.B']


In [14]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.10,
    random_state=42
)

print("Training:", X_train.shape)
print("Testing :", X_test.shape)

Training: (18006, 651)
Testing : (2001, 651)


In [15]:
elastic_cv = Pipeline([
    ("scaler", StandardScaler()),
    ("model", ElasticNetCV(
        l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9],
        alphas=np.logspace(-3, 2, 50),
        cv=5,
        max_iter=10000,
        n_jobs=-1,
        random_state=42
    ))
])

elastic_cv.fit(X_train, y_train)

best_elastic = elastic_cv.named_steps["model"]

print("Best alpha:", best_elastic.alpha_)
print("Best l1_ratio:", best_elastic.l1_ratio_)
print("Non-zero coefficients:", np.sum(best_elastic.coef_ != 0))

/home/okadeeb/miniconda3/envs/spades/lib/python3.14/site-packages/sklearn/linear_model/_coordinate_descent.py:825: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.012730e+01, tolerance: 9.591e+00
  model = cd_fast.enet_coordinate_descent_gram(
/home/okadeeb/miniconda3/envs/spades/lib/python3.14/site-packages/sklearn/linear_model/_coordinate_descent.py:825: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.944553e+01, tolerance: 9.563e+00
  model = cd_fast.enet_coordinate_descent_gram(
/home/okadeeb/miniconda3/envs/spades/lib/python3.14/site-packages/sklearn/linear_model/_coordinate_descent.py:825: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of

Best alpha: 0.006551285568595509
Best l1_ratio: 0.9
Non-zero coefficients: 352


/home/okadeeb/miniconda3/envs/spades/lib/python3.14/site-packages/sklearn/linear_model/_coordinate_descent.py:840: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.825833e+01, tolerance: 1.192e+01
  model = cd_fast.enet_coordinate_descent(


In [16]:
y_pred = elastic_cv.predict(X_test)

r2 = r2_score(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred) ** 0.5
pearson_r, pearson_p = pearsonr(y_test, y_pred)

print("R²:", r2)
print("RMSE:", rmse)
print("Pearson r:", pearson_r)
print("Pearson p:", pearson_p)

R²: 0.38484340111825655
RMSE: 1.999533547007734
Pearson r: 0.6211662303727559
Pearson p: 6.918671980860756e-214


In [10]:
from sklearn.base import clone
from sklearn.utils import resample
import time

N_BOOTSTRAPS = 100   # start with 100, not 500

alpha = best_elastic.alpha_
l1_ratio = best_elastic.l1_ratio_

boot_results = []

start = time.time()

for i in range(N_BOOTSTRAPS):
    X_boot, y_boot = resample(
        X,
        y,
        replace=True,
        random_state=42 + i
    )

    X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(
        X_boot,
        y_boot,
        test_size=0.10,
        random_state=42 + i
    )

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("model", ElasticNet(
            alpha=alpha,
            l1_ratio=l1_ratio,
            max_iter=20000,
            random_state=42 + i
        ))
    ])

    model.fit(X_train_b, y_train_b)
    pred = model.predict(X_test_b)

    boot_results.append({
        "iteration": i + 1,
        "pearson": pearsonr(y_test_b, pred)[0],
        "r2": r2_score(y_test_b, pred),
        "rmse": mean_squared_error(y_test_b, pred) ** 0.5
    })

    if (i + 1) % 10 == 0:
        elapsed = time.time() - start
        print(f"{i+1}/{N_BOOTSTRAPS} done | elapsed: {elapsed/60:.1f} min")

boot_df = pd.DataFrame(boot_results)
display(boot_df.describe())

10/100 done | elapsed: 23.4 min
20/100 done | elapsed: 46.0 min


KeyboardInterrupt: 

In [20]:
coef_df = pd.DataFrame({
    "motif": feature_cols,
    "coefficient": best_elastic.coef_
})

coef_df["abs_coefficient"] = coef_df["coefficient"].abs()

coef_df = coef_df.sort_values("abs_coefficient", ascending=False)

print("Non-zero coefficients:", (coef_df["coefficient"] != 0).sum())
display(coef_df.head(30))

Non-zero coefficients: 352


,motif,coefficient,abs_coefficient
311,ZFX.H13CORE.0.P.B,0.336073,0.336073
553,ZNF292.H13CORE.0.PSG.A,-0.316095,0.316095
220,SP140.H13CORE.0.PSGIB.A,0.291954,0.291954
35,DNTTIP1.H13CORE.0.PSG.A,-0.256578,0.256578
72,GTF2IRD2.H13CORE.0.PS.A,0.234683,0.234683
140,NFIC.H13CORE.1.PSM.A,-0.229332,0.229332
57,GABPA.H13CORE.0.PSM.A,0.204216,0.204216
159,NRF1.H13CORE.0.PS.A,0.194705,0.194705
560,ZNF35.H13CORE.1.P.C,0.177159,0.177159
257,TYY1.H13CORE.0.PSM.A,0.173565,0.173565


In [18]:
boot_df = pd.DataFrame(boot_results)
display(boot_df.describe())

,iteration,pearson,r2,rmse
count,25.000000,25.000000,25.000000,25.000000
mean,13.000000,0.638013,0.406604,1.977410
std,7.359801,0.015951,0.020467,0.041684
min,1.000000,0.613744,0.376232,1.915633
25%,7.000000,0.624656,0.389950,1.944437
50%,13.000000,0.632099,0.398799,1.973363
75%,19.000000,0.647914,0.419091,2.000595
max,25.000000,0.670479,0.449282,2.049658


In [24]:
coef_df.to_csv("../results/H13_ElasticNet_coefficients.csv", index=False)

print("Saved", len(coef_df), "motif coefficients.")

Saved 651 motif coefficients.


In [23]:
mkdir -p data/processed
mkdir -p results
mkdir -p figures

SyntaxError: invalid syntax (4215468260.py, line 1)